# Business Problem
An Online travel booking company is suffering from loss in revenue because of the uncertain booking cancelation of its customers. The company wants to know which customer will cancel the booking.

# Dataset Detailes

0   hotel (H1 = Resort Hotel or H2 = City Hotel)

1   is_canceled Value indicating if the booking was canceled (1) or not (0)

2   lead_time Number of days that elapsed between the entering date of the booking into the PMS and the arrival date

3   arrival_date_year Year of arrival date

4   arrival_date_month Month of arrival date

5   arrival_date_week_number Week number of year for arrival date

6   arrival_date_day_of_month Day of arrival date

7   stays_in_weekend_nights Number of weekend nights (Saturday or Sunday) the guest stayed or booked to stay at the hotel

8   stays_in_week_nights Number of week nights (Monday to Friday) the guest stayed or booked to stay at the hotel

9   adults Number of adults

10  children Number of children

11  babies Number of babies

12  meal Type of meal booked. Categories are presented in standard hospitality meal packages: Undefined/SC – no meal

13  country Country of origin. Categories are represented in the ISO 3155–3:2013 format

14  market_segment Market segment designation. In categories, the term “TA” means “Travel Agents” and “TO” means “Tour Operators”

15  distribution_channel Booking distribution channel. The term “TA” means “Travel Agents” and “TO” means “Tour Operators”

16  is_repeated_guest Value indicating if the booking name was from a repeated guest (1) or not (0)

17  previous_cancellations Number of previous bookings that were cancelled by the customer prior to the current booking

18  previous_bookings_not_canceled Number of previous bookings not cancelled by the customer prior to the current booking

19  reserved_room_type Code of room type reserved. Code is presented instead of designation for anonymity reasons.

20  assigned_room_typeCode for the type of room assigned to the booking.Code is presented instead of designation for anonymity reasons.

21  booking_changes Number of changes made to the booking from the moment the booking was entered on the PMS until the moment of check-in or out

22  deposit_type Indication on if the customer made a deposit to guarantee the booking. This variable can assume three categories: No

23  agent ID of the travel agency that made the booking

24  company ID of the company that made the booking or responsible for paying the booking.

25  days_in_waiting_list Number of days the booking was in the waiting list before it was confirmed to the customer

26  customer_type Type of booking, assuming one of four categories:Transient - Transient-Party - Contract - Group

27  adr Average Daily Rate as defined by dividing the sum of all lodging transactions by the total number of staying nights

28  required_car_parking_spaces Number of car parking spaces required by the customer

29  total_of_special_requestsNumber of special requests made by the customer (e.g. twin bed or high floor)

30  reservation_status Reservation last status, assuming one of three categories: Canceled – booking was canceled by the customer; Check-Out

31  reservation_status_date Date at which the last status was set. This variable can be used in conjunction with the ReservationStatus to

# Importing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import folium
from folium.plugins import HeatMap
import plotly.express as px

!pip install sort-dataframeby-monthorweek
!pip install sorted-months-weekdays


In [ ]:
#Load data
data=pd.read_csv("/kaggle/input/hotel-booking-demand/hotel_bookings.csv")
data.head(10)

In [ ]:
data.shape

# Data Analysis

In [ ]:
data.describe().T

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data['country'].fillna('PRT',inplace=True)
data.fillna(0,inplace=True)
data.isnull().sum()

In [ ]:
# adults, babies and children cant be zero at same time, so dropping the rows having all these zero at same time

filter = (data.children == 0) & (data.adults == 0) & (data.babies == 0)
data[filter]

In [ ]:
is_can = len(data[data['is_canceled']==1])
print("Percentage cancelation= ", is_can/len(data))
data['reservation_status'].value_counts(normalize=True)*100

Number of guest in each hotels

# Exploratory Data Analysis (EDA)

In [ ]:
resort_len= len(data[data['hotel']=='Resort Hotel'])
city_len=len(data[data['hotel']=='City Hotel'])

hotel_types = ['Resort Hotel', 'City Hotel']
lengths = [resort_len, city_len]

plt.bar(hotel_types, lengths)
plt.xlabel('Hotel Type')
plt.ylabel('Number of Guest')
plt.title('Number of Guest for Each Hotel Type')
plt.show()


In [ ]:
country_wise_counts = data[data['is_canceled'] == 0]['country'].value_counts().reset_index()
country_wise_counts.columns = ['country', 'No of guests']
country_wise_counts

In [ ]:
basemap = folium.Map()
guests_map = px.choropleth(country_wise_counts, locations = country_wise_counts['country'],
                           color = country_wise_counts['No of guests'], hover_name = country_wise_counts['country'])
guests_map.show()

Most guests are from Portugal and other countries in Europe.

In [ ]:
df = data[data['is_canceled'] == 0]
px.box(data_frame = df, x = 'reserved_room_type', y = 'adr', color = 'hotel', template = 'plotly_dark')

The figure shows that avgerage price per room

In [ ]:
data_resort = data[(data['hotel'] == 'Resort Hotel') & (data['is_canceled'] == 0)]
data_city = data[(data['hotel'] == 'City Hotel') & (data['is_canceled'] == 0)]

In [ ]:
resort_hotel = data_resort.groupby(['arrival_date_month'])['adr'].mean().reset_index()
resort_hotel

In [ ]:
city_hotel = data_city.groupby(['arrival_date_month'])['adr'].mean().reset_index()
city_hotel

In [ ]:
final_hotel = resort_hotel.merge(city_hotel, on = 'arrival_date_month')
final_hotel.columns = ['month', 'price_for_resort', 'price_for_city_hotel']
final_hotel

In [ ]:
import sort_dataframeby_monthorweek as sd

def sort_month(df, column_name):
    return sd.Sort_Dataframeby_Month(df, column_name)

In [ ]:
final_prices = sort_month(final_hotel, 'month')
final_prices

In [ ]:
plt.figure(figsize = (19, 10))

px.line(final_prices, x = 'month', y = ['price_for_resort','price_for_city_hotel'],
        title = 'Room price per night over the Months', template = 'plotly_dark')

Resort_Hotel- expensive in augest month  //  
City_Hotel- expensive in may month

In [ ]:
filter=data["is_canceled"]==0
df=data[filter]
df.head()

In [ ]:
df['total_night']=df['stays_in_weekend_nights']+df['stays_in_week_nights']
df

In [ ]:
stay = df.groupby(['total_night', 'hotel']).agg('count').reset_index()
stay = stay.iloc[:, :3]
stay = stay.rename(columns={'is_canceled':'Number of stays'})
stay

In [ ]:
px.bar(data_frame = stay, x = 'total_night', y = 'Number of stays', color = 'hotel', barmode = 'group',
        template = 'plotly_dark')

In [ ]:
plt.figure(figsize=(10,8))
sns.barplot(x=data[data['is_canceled']==0].groupby('market_segment')['stays_in_weekend_nights'].count().index,
            y=data[data['is_canceled']==0].groupby('market_segment')['stays_in_weekend_nights'].count())

# Data Preprocessing

In [ ]:
plt.figure(figsize = (28, 14))

corr = data.corr()
sns.heatmap(corr, annot = True, linewidths = 1)
plt.show()

In [ ]:
data.columns

In [ ]:
data = data.drop_duplicates()
len(df)

In [ ]:
label = ['company','agent','total_of_special_requests','required_car_parking_spaces','booking_changes',
         'is_repeated_guest','reservation_status_date','stays_in_weekend_nights','stays_in_week_nights',
         'reserved_room_type','assigned_room_type','adults','children','babies']
data.drop(labels=label,axis=1,inplace=True)
data

In [ ]:
X = data.drop(['is_canceled'],axis=1)
y = data['is_canceled']

In [ ]:
X = pd.get_dummies(X,drop_first=True)


In [ ]:
X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression

# LogisticRegression Model

In [ ]:
# LogisticRegression
log = LogisticRegression(solver='liblinear')
log.fit(X_train,y_train)
#Print the model accuracy of training data
print('Logistic Regression Training Accuracy : ',log.score(X_train, y_train))

In [ ]:
cm = confusion_matrix(y_test,log.predict(X_test))
print(cm)
print('Accuracy ',accuracy_score(y_test,log.predict(X_test)))

# 100% Accuracy

In [ ]:
print(classification_report(y_test,log.predict(X_test)))